<a href="https://colab.research.google.com/github/alanna-jc/AutoEncoder_Anomaly_Detector/blob/Beta-VAE/VAE_Anomaly_Detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Beta-VAE Anomaly Detector

Loss Fun: MSE + Beta * (KL Divergence)

Beta-Variational: Encoder outputs a mean and a log variance. A vector is sampled from gaussian noise, and said vector is scaled and shifted by the mean and std from the encoder. This vector is the latent vector we pass our Decoder.

A hyper parameter beta is added to scale how much the KL divergence affects the loss function


In [ ]:
# Download the dataset, setup packages
import os
import cv2
import numpy as np
import numpy.typing as npt
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, roc_auc_score

import torch as torch
import torch.nn as nn
import torch.optim as optim

if not os.path.exists('dataset.zip'):
  !gdown 1_pRKXtYRjWjY0seYqyx25nOxjtr-mHYg
  !unzip -q -u dataset.zip
else:
  print('Already downloaded')

Downloading...
From: https://drive.google.com/uc?id=1_pRKXtYRjWjY0seYqyx25nOxjtr-mHYg
To: /content/dataset.zip
100% 2.60M/2.60M [00:00<00:00, 135MB/s]


In [ ]:
# Some helper functions
def load_dataset(class_name = 'pasta'):
  assert class_name in ['pasta', 'screws']
  dir = './dataset/'+class_name+'/'
  training_images = []
  testing_images = []
  testing_labels = []
  for file_name in os.listdir(dir+'train/good/'):
    training_images.append(cv2.cvtColor(cv2.imread(dir+'train/good/'+file_name), cv2.COLOR_BGR2RGB))
  for file_name in os.listdir(dir+'test/good/'):
    testing_images.append(cv2.cvtColor(cv2.imread(dir+'test/good/'+file_name), cv2.COLOR_BGR2RGB))
    testing_labels.append(0)
  for file_name in os.listdir(dir+'test/bad/'):
    testing_images.append(cv2.cvtColor(cv2.imread(dir+'test/bad/'+file_name), cv2.COLOR_BGR2RGB))
    testing_labels.append(1)

  # returns a normalized (0-1) numpy array of size (n,)
  return np.array(training_images)/255., np.array(testing_images)/255., np.array(testing_labels)

def basic_evaluation(predictions : np.ndarray, targets : np.ndarray):
  print(targets)
  print(predictions)
  print('AUROC Score:', roc_auc_score(targets, predictions))

In [ ]:
# What size to make latent vector dim?
class VAE_Encoder(nn.Module):
    def __init__(self, image_height, image_width, latent_vector_dim = 128):
        super().__init__()

        '''
        Encoder has 3 3x3 convolutional layers which reduces dimensions in half each layer (stride = 2). These layers act as feature extractors

        Recall: stride is how many pixels filter slides (produces less outputs for bigger stride). It is 'learned downsampling'.
                In contrast pooling throws away backgroud noise and has no 'learning'
        '''

        self.convolution = nn.Sequential(
            # (in_channels, out_channels, kernel_size, stride, padding)
            nn.Conv2d(3,16,3,stride=2,padding=1),
            nn.ReLU(),
            nn.Conv2d(16,32,3,stride=2,padding=1),
            nn.ReLU(),
            nn.Conv2d(32,64,3,stride=2,padding=1),
            nn.ReLU()
        )

        '''
        We will want to flatten our output vector and then compute the mean and log variance for 'Variational' aspect of autoencoder
        The mean and variance are learned parameters. Given this, it is how they are used and their affect on the loss function that matters,
        not how they are defined.

        Essentialy: it appears we do not calculate these values, they instead rise to their calling

        To get these values from latent vector, we use fully connected layer to scale down to 1 param (1 meam and 1 variance) (as mean and variance use all values in vector?)
        Options are nn.Linear(input_dim, output_dim) or nn.LazyLinear(output_dim) which does it for you?

        Using linear as we need these values anyways for our decoder. therefore we have to do a dummy pass to see the output shapes
        '''
        dummy_input = torch.zeros(1, 3, image_height, image_width)
        dummy_output = self.convolution(dummy_input)

        # Save this shape for the decoder to use later
        self.flattened_size = dummy_output.view(1, -1).size(1)
        self.spatial_shape = dummy_output.shape[1:]

        self.fully_connected_mu = nn.Linear(self.flattened_size, latent_vector_dim)
        self.fully_connected_log_var = nn.Linear(self.flattened_size, latent_vector_dim)

        # # fully connected layer
        # self.fully_connected_mu = nn.LazyLinear(latent_vector_dim)
        # # fully connected layer
        # self.fully_connnected_log_var = nn.LazyLinear(latent_vector_dim)

    def forward(self,x):

        # Go through our convolutional layers
        z = self.convolution(x)

        # flatten output into latent vector
        # if doing 1x1x1 convolution you do not flatten you jsut march on

        x = torch.flatten(z, start_dim =1)

        # change to convolutional 1x1x1
        # now we want our flattened vector to pass through both fc layers to yeild 2 parameters which will act as our mean and variance
        mu = self.fully_connected_mu(x)
        log_var = self.fully_connected_log_var(x)

        return mu, log_var


In [ ]:
# put reparam here as separate helper or what
def reparametrization(mu, log_var):
      '''
      Here I want to create gaussian noise, select something random from there and do a shift and scale for mu and logvar
      '''
      # std = sqrt(variance)
      var = torch.exp(log_var)
      std = torch.sqrt(var)
      # torch.randn_like returns a tensor with same size as input of a random value from a gaussian distribution with mean 0 and std 1
      # inputting in std as want to have same shapw
      eps = torch.randn_like(std)
      # scale and shift are gaussian by the mu and std
      # z = mean + (random noise * standard deviation)
      z = mu + (eps *std)

      return z

In [ ]:
class VAE_Decoder(nn.Module):
    def __init__(self, latent_vector_dim, flattened_size, spatial_shape):
        super().__init__()
        self.spatial_shape = spatial_shape

        self.fully_connected_decoder_in = nn.Linear(latent_vector_dim, flattened_size)

        # decoder mirrors encoder and reverses to reconstruct the spacial resolution
        self.decoder = nn.Sequential(
            nn.Unflatten(1,self.spatial_shape),
            nn.ConvTranspose2d(64,32,4,stride=2,padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32,16,4,stride=2,padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(16,3,4,stride=2,padding=1),
            nn.Sigmoid()
        )


    def forward(self, z):

        # expand the latent vector into an input big enough for our decoder
        x = self.fully_connected_decoder_in(z)
        output = self.decoder(x)
        return output

In [ ]:
def VAE_loss(x, recon_out, mu, log_var, beta = 0.01):
  # Reconstruction loss (MSE)
  reconstruction_loss = nn.MSELoss()(recon_out, x)

  #KL loss (ai wrote this for me :) )
  kl_loss = -0.5 * torch.mean(
        1 + log_var - mu.pow(2) - log_var.exp())

  # output L = recon_loss + kl_loss
  return reconstruction_loss + (beta*kl_loss)

In [ ]:
# TODO This class will be the class to modify.
# The current implementation of this model tries to take the median of all the
# Training images and then measure the L2 distance between this median and any
# test images.
class VAE(nn.Module):
  def __init__(self, image_height, image_width, latent_vector_dim = 128, num_epochs=100):
    super().__init__()

    self.encoder = VAE_Encoder(image_height, image_width)

    self.decoder = VAE_Decoder(latent_vector_dim, self.encoder.flattened_size, self.encoder.spatial_shape)

  def forward(self, x):
    mu, logvar = self.encoder(x)
    z = reparametrization(mu, logvar)
    reconstructed_output = self.decoder(z)

    return reconstructed_output, mu, logvar



In [ ]:
def train_model(model, dataset, num_epochs, learning_rate=1e-3, beta=0.01):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    # convert numpy to torch, need to rearrange the dimensions
    x = torch.tensor(dataset).permute(0,3,1,2).float().to(device)

    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    for epoch in range(num_epochs):

      model.train()

      # compute vae forward
      recon_out, mu, logvar = model(x)

      # compute loss
      loss = VAE_loss(x, recon_out, mu, logvar, beta)

      # clear old gradients from last iter
      optimizer.zero_grad()

      # backward pass for optimization
      loss.backward()

      # update
      optimizer.step()

      print("epoch",epoch,"loss",loss.item())


def predict(model, test_data):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    model.eval()

    x = torch.tensor(test_data).permute(0,3,1,2).float().to(device)

    with torch.no_grad():

      x_hat, _, _ = model(x)

      errors = torch.mean((x - x_hat)**2, dim=(1,2,3))

    return errors.cpu().numpy()


In [ ]:
# TODO use class above as well as helper functions to generate
# predictions on the datasets and evaluate the results.

def do_analysis(class_name):
  training_images, testing_images, testing_labels = load_dataset(class_name=class_name)

  img_height = training_images.shape[1]
  img_width = training_images.shape[2]

  vae_model = VAE(img_height, img_width)

  train_model(vae_model, training_images, num_epochs=200, beta = 0.01)

  predictions = predict(vae_model, testing_images)
  basic_evaluation(predictions, testing_labels)

print("screws analysis")
do_analysis('screws')
print("pasta analysis")
do_analysis('pasta')


screws analysis
epoch 0 loss 0.07021041959524155
epoch 1 loss 0.07180967181921005
epoch 2 loss 0.068390391767025
epoch 3 loss 0.06800296902656555
epoch 4 loss 0.06736745685338974
epoch 5 loss 0.06600397825241089
epoch 6 loss 0.06456360965967178
epoch 7 loss 0.06304946541786194
epoch 8 loss 0.06152033805847168
epoch 9 loss 0.05968732014298439
epoch 10 loss 0.05772949010133743
epoch 11 loss 0.055150751024484634
epoch 12 loss 0.05229175090789795
epoch 13 loss 0.04938799887895584
epoch 14 loss 0.04574332386255264
epoch 15 loss 0.0428973063826561
epoch 16 loss 0.03856666386127472
epoch 17 loss 0.03481622785329819
epoch 18 loss 0.0318368598818779
epoch 19 loss 0.030258608981966972
epoch 20 loss 0.029089469462633133
epoch 21 loss 0.02819802798330784
epoch 22 loss 0.02755816839635372
epoch 23 loss 0.02731778845191002
epoch 24 loss 0.027614939957857132
epoch 25 loss 0.02639586478471756
epoch 26 loss 0.027353564277291298
epoch 27 loss 0.026229027658700943
epoch 28 loss 0.025360170751810074
epoch